# One-region parameter simulation


<details>
<summary>References</summary>

* https://github.com/the-virtual-brain/tvb-root/blob/master/tvb_library/tvb/simulator/models/zerlaut.py

* https://github.com/the-virtual-brain/tvb-root/blob/master/tvb_library/tvb/simulator/simulator.py

</details>

## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.


In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
import sys
from pathlib import Path

defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }

os.environ.update(settings)
import_dir = Path(settings.get('TVB_IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))


Modules

In [ ]:
import json5 as json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from dataclasses import asdict, dataclass #, field

In [ ]:
from tvb.basic.neotraits.api import List, NArray
from tvb.datatypes.connectivity import Connectivity
from tvb.simulator.coupling import Linear

from tvb.simulator.monitors import Raw
from tvb.simulator.simulator import Simulator

In [ ]:
#from tvb.simulator.integrators import HeunDeterministic, HeunStochastic
#from tvb.simulator.noise import Additive #, Multiplicative
#from tvb.simulator.models.zerlaut import ZerlautAdaptationFirstOrder
from tvb.simulator.models import ModelsEnum
from tvb.simulator import noise as Noise, integrators as Integrators

Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:
from src.environment import resolve_directory

#ToDo: Cambiar el nombe de las variables de entorno
WORK_DIR = resolve_directory(settings['TVB_WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Entradas de la simulación.

# 1.0) Salida de la simulación.
OUTPUT_DIR = resolve_directory(settings['TVB_RESULTS_DIR'], WORK_DIR)
OUTPUT_FILE = 'single_region_simulation'
OUTPUT_DTYPE = "float32"

# OUTPUT_STATES = None
# OUTPUT_STATES = ["E_e", "E_i"]
OUTPUT_STATES = ['E', 'I', 'W_e', 'W_i', 'ou_drift']

# Solo requerido para tareas como plotting.
STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}

STIMULUS_UNITS = {
    "external_input_ex_ex": "kHz",
    "external_input_ex_in": "kHz",
    "external_input_in_ex": "kHz",
    "external_input_in_in": "kHz",
}


os.makedirs(OUTPUT_DIR, exist_ok=True)
pprint(f'Results directory: {OUTPUT_DIR}')

In [ ]:
# 1.1) Parámetros de la simulación.
DURATION_MS = 2000.0    # Duración de la simulación en milisegundos.
DT_MS = 0.5             # Paso de integración en milisegundos.

# INTEGRATOR_NAME = 'HeunStochastic'  
NOISE ={
    "name": "Additive",
    "params": {
        "nsig": [0, 0, 0, 0, 1]
    }
}


In [ ]:
# 1.2) Parámetros de la entrada. En este caso un pulso.
STIMULUS = {
    "external_input_ex_ex": {
        "pulse": {
            "baseline_khz": 0.005,        # Frecuencia de disparo basal
            "pulse_amplitude_khz": 0.04,  # Amplitud del pulso
            "pulse_onset_ms": 500.0,      # Instante de inicio del pulso
            "pulse_duration_ms": 800.0,   # Duración del pulso
        },
    },
    # # Define los que vas a utilizar
    # "external_input_ex_in": {
    #     "pulse_train": {
    #             "baseline_khz": 0.001,        # Frecuencia de disparo basal
    #             "pulse_amplitude_khz": 0.01,  # Amplitud del pulso
    #             "pulse_onset_ms": 500.0,      # Instante de inicio del pulso
    #             "pulse_duration_ms": 100.0,   # Duración del pulso
    #             "pulse_period_ms": 150.0,     # Periodo del tren de pulsos
    #         },
    # }, 
    
    # "external_input_in_ex": {"external_input_ex_ex": 0.5},
    # "external_input_in_in": {"external_input_ex_in": 0.5},
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"


MODEL_PARAMETERS = {
    # Población y conectividad
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Parametros de membrana
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Parametors de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 20.0,
    "tau_w_i": 100.0,

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    "T": 10.0,

    # Parametro de segundo orden (no hay un motivo matemático)
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Parametros de ruido
    "weight_noise": 2e-4, #10.5
    "tau_OU": 2, #0.5
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


**Parametros que se utilizaran para estimular el modelo.**

```python
[external_input_ex_ex, external_input_ex_in, 
external_input_in_ex, external_input_in_in]
```

**Parametros de ruido.**

<font color=#fda> NOTE: </font> Vamos a utilizar los valores aleatorios para generar ruido,como aumento de datos. 

```python
[tau_OU, weight_noise]
```

## Build the stimulus

In [ ]:
from src.tvb.stimulus import _build_pulse


In [ ]:
from src.tvb.stimulus import _build_pulse_train


In [ ]:
# ToDo: Esta clase devuelve arrays, pero la clase del simulador acepta un diccionario de funciones. 

from src.tvb.stimulus import build_stimulus


In [ ]:
time_ms, stimulus = build_stimulus(STIMULUS, DURATION_MS, DT_MS)

In [ ]:
# Plotting

fig, ax = plt.subplots(figsize=(10, 3))

for key, values in stimulus:
    ax.plot(time_ms, values, label=key)

ax.set_title("Stimulation Signals")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Stimulus (KHz)")
ax.legend()
plt.show()

## Auxiliary functions

### New simulator class

In [ ]:
### New Simulation Class
# ToDo: Escoger nombre. DynamicParameterSimulator, ParameterStimulationSimulator,
#       TimeVaryingParameterSimulator

from src.tvb.simulator import DynamicParameterSimulator


### Build simulation components

In [ ]:
from src.tvb.components import get_model_class, get_integrator_class, get_noise_class

# model_name = "ZerlautAdaptationFirstOrder"

In [ ]:
from src.tvb.components import build_model

model = build_model(MODEL_PARAMETERS | {"weight_noise": 0.0}, INTEGER_PARAMETERS)
pprint(f'Model variables of interest: {model.variables_of_interest}')
pprint(f'Model state variables: {model.state_variables}')
pprint(f'Model coupling variables: {np.asarray(model.state_variables)[model.cvar]}')
pprint(f'Model stimulus variables: {model.stvar}') # None => cvar


In [ ]:
from src.tvb.components import build_integrator

# ToDo: Parametros del integrador. Comprobar en el futuro
# integrator_class = get_integrator_class("HeunStochastic")
# for name in integrator_class.declarative_attrs:
#     pprint(name, repr(getattr(integrator_class, name)))

# noise_class = get_noise_class("Additive")
# for name in noise_class.declarative_attrs:
#     print(name, repr(getattr(noise_class, name)))

In [ ]:
from src.tvb.simulator import build_single_region_simulator


***Listar las opciones: Modelos, integradores y ruido.***

In [ ]:
# # Posibles modelos
# pprint(f"Modelos disponibles: {sorted(model.value for model in ModelsEnum)}")


# # stoch = {n[:-10] for n in Integrators.__dict__ if n.upper().endswith('STOCHASTIC')}
# # deter = {n[:-13] for n in Integrators.__dict__ if n.upper().endswith('DETERMINISTIC')}
# # integrators = stoch | deter
# # # Seguro que todos son integradores
# # pprint(f"Integradores disponibles: {sorted(integrators)}")
# # No todos son ingradores
# # pprint(f"Integradores disponibles: {[n for n in Integrators.__dict__] }")

# integrator_names = sorted(
#     name for name, cls in vars(Integrators).items()
#     if isinstance(cls, type)
#     and issubclass(cls, Integrators.Integrator)
# )
# pprint(f"Integradores disponibles: {integrator_names}") # HeunDeterministic, HeunStochastic


# noise_names = [
#     name for name, cls in vars(Noise).items()
#     if isinstance(cls, type)
#     and issubclass(cls, Noise.Noise)
# ]
# pprint(f"Ruido disponible: {noise_names}") # Additive, Multiplicative


### Save and load resuls


In [ ]:
from src.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata




In [ ]:
from src.tvb.results import save_single_region_simulation


In [ ]:
from src.tvb.results import load_results


### Plot results

In [ ]:
from src.tvb.plotting import plot_results


## Deterministic Simulation

### Run simulaiton

In [ ]:
# integrator = HeunStochastic(dt=0.1, noise=Additive(nsig=np.array([0,0,0,0,0.01])))
# integrator = HeunDeterministic(dt=0.1)

integrator = build_integrator(DT_MS, integrator_class="HeunDeterministic")# , noise_config=NOISE)

In [ ]:

model = build_model(
    MODEL_PARAMETERS, INTEGER_PARAMETERS, 
    model_class="ZerlautAdaptationFirstOrder",
    variable_of_interest=OUTPUT_STATES
)

simulator = build_single_region_simulator(model, stimulus, integrator)

pprint('Configured sources:', [(name, kind) for name, _, kind in simulator._parameter_sources])

In [ ]:
time_detsim_raw, detsim_raw = simulator.run(n_steps=len(time_ms))[0] 
detsim_output = detsim_raw[:, :, 0, 0]  # history, state, region, node,

pprint(f"Simulation output shape: {detsim_output.shape}")

In [ ]:
pprint(f"Simulation steps: {time_detsim_raw.shape}")
pprint(f"Stimulation steps: {time_ms.shape}")

#Ojo parece que hay un desfase y esto puede ser un problema.
pprint(f"Maximum difference: {np.max(np.abs(time_detsim_raw - time_ms))}")
pprint(f"Minimum difference: {np.min(np.abs(time_detsim_raw - time_ms))}")

### Save results

In [ ]:
output_variables = list(model.variables_of_interest if OUTPUT_STATES is None else OUTPUT_STATES)
selected_output = detsim_output[:, [model.variables_of_interest.index(name) for name in output_variables]]

metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(
        name=model,
        parameters=MODEL_PARAMETERS,
        integer_parameters=list(INTEGER_PARAMETERS), # No puedes descargar sets
        output_variables=output_variables,
        state_units=STATE_UNITS,
        stimulus_units=STIMULUS_UNITS,
    ),
    stimulus=STIMULUS,
)

OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_FILE}_det1.npz"
save_single_region_simulation(
    OUTPUT_PATH, time_detsim_raw, selected_output, metadata,
)

pprint(f"Saved: '{OUTPUT_PATH}'")


### Load results

In [ ]:
loaded_results = load_results(OUTPUT_PATH)
pprint(f"Loaded {len(loaded_results['time_ms'])} samples from '{OUTPUT_PATH}'")
pprint("State variables:", list(loaded_results["states"]))
pprint("Metadata:", type(loaded_results["metadata"]).__name__)


### Plot results


In [ ]:
PLOT_STATES = None  # Por ejemplo: ["E", "W_e"]

fig, axes = plot_results(
    loaded_results["time_ms"],
    loaded_results["states"],
    loaded_results["metadata"],
    show_stimulus=True,
    plot_states=PLOT_STATES,
)
plt.show()


## Stochastic Simulation

### Run simulaiton

In [ ]:
# integrator = HeunStochastic(dt=0.1, noise=Additive(nsig=np.array([0,0,0,0,0.01])))
# integrator = HeunDeterministic(dt=0.1)

NOISE |= {
    "name": "Additive",
    "params": {
        "nsig": [0, 0, 0, 0, 1]
    }
}

MODEL_PARAMETERS |= {
    "weight_noise": 2e-4, #10.5
    "tau_OU": 0.5 #2, #0.5
}

integrator = build_integrator(DT_MS, integrator_class="HeunStochastic", noise_config=NOISE)

In [ ]:

model = build_model(
    MODEL_PARAMETERS, INTEGER_PARAMETERS, 
    model_class="ZerlautAdaptationFirstOrder",
    variable_of_interest=OUTPUT_STATES
)

simulator = build_single_region_simulator(model, stimulus, integrator)

pprint('Configured sources:', [(name, kind) for name, _, kind in simulator._parameter_sources])

In [ ]:
time_detsim_raw, detsim_raw = simulator.run(n_steps=len(time_ms))[0] 
detsim_output = detsim_raw[:, :, 0, 0]  # history, state, region, node,

pprint(f"Simulation output shape: {detsim_output.shape}")

In [ ]:
pprint(f"Simulation steps: {time_detsim_raw.shape}")
pprint(f"Stimulation steps: {time_ms.shape}")

#Ojo parece que hay un desfase y esto puede ser un problema.
pprint(f"Maximum difference: {np.max(np.abs(time_detsim_raw - time_ms))}")
pprint(f"Minimum difference: {np.min(np.abs(time_detsim_raw - time_ms))}")

### Save results

In [ ]:
output_variables = list(model.variables_of_interest if OUTPUT_STATES is None else OUTPUT_STATES)
selected_output = detsim_output[:, [model.variables_of_interest.index(name) for name in output_variables]]

metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(
        name=model,
        parameters=MODEL_PARAMETERS,
        integer_parameters=list(INTEGER_PARAMETERS), # No puedes descargar sets
        output_variables=output_variables,
        state_units=STATE_UNITS,
        stimulus_units=STIMULUS_UNITS,
    ),
    stimulus=STIMULUS,
)

OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_FILE}_stoch1.npz"
save_single_region_simulation(
    OUTPUT_PATH, time_detsim_raw, selected_output, metadata,
)

pprint(f"Saved: '{OUTPUT_PATH}'")


### Load results

In [ ]:
loaded_results = load_results(OUTPUT_PATH)
pprint(f"Loaded {len(loaded_results['time_ms'])} samples from '{OUTPUT_PATH}'")
pprint("State variables:", list(loaded_results["states"]))
pprint("Metadata:", type(loaded_results["metadata"]).__name__)


### Plot results


In [ ]:
PLOT_STATES = None  # Por ejemplo: ["E", "W_e"]

fig, axes = plot_results(
    loaded_results["time_ms"],
    loaded_results["states"],
    loaded_results["metadata"],
    show_stimulus=True,
    plot_states=PLOT_STATES,
)
plt.show()


## Second Order Stochastic Simulation

### Run simulaiton

In [ ]:
# integrator = HeunStochastic(dt=0.1, noise=Additive(nsig=np.array([0,0,0,0,0.01])))
# integrator = HeunDeterministic(dt=0.1)

NOISE |= {
    "name": "Additive",
    "params": {
        "nsig": [0, 0, 0, 0, 0, 0, 0, 1]
    }
}

MODEL_PARAMETERS |= {
    "weight_noise": 2e-4, #10.5
    "tau_OU": 0.5 #2, #0.5
}

OUTPUT_STATES += ["C_ee", "C_ei", "C_ii"]
OUTPUT_STATES = list(set(OUTPUT_STATES))  # Eliminar duplicados

integrator = build_integrator(DT_MS, integrator_class="HeunStochastic", noise_config=NOISE)

In [ ]:
model = build_model(
    MODEL_PARAMETERS, INTEGER_PARAMETERS, 
    model_class="ZerlautAdaptationSecondOrder",
    variable_of_interest= OUTPUT_STATES
)

simulator = build_single_region_simulator(model, stimulus, integrator)

pprint('Configured sources:', [(name, kind) for name, _, kind in simulator._parameter_sources])
pprint(f'Model variables of interest: {model.variables_of_interest}')
pprint(f'Model state variables: {model.state_variables}')

In [ ]:
time_detsim_raw, detsim_raw = simulator.run(n_steps=len(time_ms))[0] 
detsim_output = detsim_raw[:, :, 0, 0]  # history, state, region, node,

pprint(f"Simulation output shape: {detsim_output.shape}")

In [ ]:
pprint(f"Simulation steps: {time_detsim_raw.shape}")
pprint(f"Stimulation steps: {time_ms.shape}")

#Ojo parece que hay un desfase y esto puede ser un problema.
pprint(f"Maximum difference: {np.max(np.abs(time_detsim_raw - time_ms))}")
pprint(f"Minimum difference: {np.min(np.abs(time_detsim_raw - time_ms))}")

### Save results

In [ ]:
output_variables = list(model.variables_of_interest if OUTPUT_STATES is None else OUTPUT_STATES)
selected_output = detsim_output[:, [model.variables_of_interest.index(name) for name in output_variables]]

metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(
        name=model,
        parameters=MODEL_PARAMETERS,
        integer_parameters=list(INTEGER_PARAMETERS), # No puedes descargar sets
        output_variables=output_variables,
        state_units=STATE_UNITS,
        stimulus_units=STIMULUS_UNITS,
    ),
    stimulus=STIMULUS,
)

OUTPUT_PATH = OUTPUT_DIR / f"{OUTPUT_FILE}_stoch1.npz"
save_single_region_simulation(
    OUTPUT_PATH, time_detsim_raw, selected_output, metadata,
)

pprint(f"Saved: '{OUTPUT_PATH}'")


### Load results

In [ ]:
loaded_results = load_results(OUTPUT_PATH)
pprint(f"Loaded {len(loaded_results['time_ms'])} samples from '{OUTPUT_PATH}'")
pprint("State variables:", list(loaded_results["states"]))
pprint("Metadata:", type(loaded_results["metadata"]).__name__)


### Plot results


In [ ]:
PLOT_STATES = None  # Por ejemplo: ["E", "W_e"]
#PLOT_STATES = ["E", "I", "W_e", "W_i", "C_ee", "C_ei", "C_ii", "ou_drift"]
PLOT_STATES = ["C_ee", "C_ei", "C_ii"]


fig, axes = plot_results(
    loaded_results["time_ms"],
    loaded_results["states"],
    loaded_results["metadata"],
    show_stimulus=True,
    plot_states=PLOT_STATES,
)
plt.show()
